1. 加载现有数据
2. 进行前置预筛选
3. 进行信息提取
4. 组织并输出

In [ ]:
import tomllib
from pathlib import Path
from loguru import logger
from cetc_product.tools.llm_factory import llm_manager
import os
import orjson
CONFIG_PATH = "/Data_two/wyw/code/CETC_product/config.toml"

In [ ]:
path = "/Data_two/wyw/code/CETC_product/data/out/task2/enwiki/20251117_151015"

In [ ]:
with open(path, "rb") as fin:
    for line in fin:
        record = orjson.loads(line)
        print(record)


In [6]:
llm_config= {
    "model": "qwen-plus",
    "model_provider": "openai",
    "temperature": 0,
    "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
    "api_key": "sk-91f5ab07b5b441488ce6c281b3c93589",
    "extra_body": {"enable_thinking": True},
    "stream": True,
    
}

In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model(**llm_config)

ImportError: Unable to import langchain_openai. Please install with `pip install -U langchain-openai`

In [ ]:
for x in model.stream("你是谁"):
    print(x,flush=True)

In [ ]:
config = tomllib.load(Path(CONFIG_PATH).open("rb"))

In [ ]:
def safe_get(d: dict, keys: list, default=None):
    """Safely get a nested value from a dictionary."""
    for key in keys:
        if isinstance(d, dict) and key in d:
            d = d[key]
        else:
            return default
    return d

In [ ]:
ip = "/Data_two/wyw/code/CETC_product/data/prompts/NER_for_org.txt"
sorted(list(Path(ip).parent.glob(Path(ip).name)))

In [ ]:
from typing import Any, Generator, Tuple
import xopen
import orjson
import orjsonl
import yaml
def read_indata(
    inpath: Path, 
    encoding: str = "utf-8",
    skip_errors: bool = True
) -> Generator[Tuple[Path, Any], None, None]:
    """
    从文件中逐行读取 JSON 数据
    
    Args:
        inpath: 输入文件路径（支持通配符）
        encoding: 文件编码
        skip_errors: 是否跳过解析错误的行
    """
    # 获取匹配的文件列表
    inpaths = sorted(list(Path(inpath.parent).glob(inpath.name)))
    
    if not inpaths:
        logger.warning(f"未找到匹配的文件: {inpath}")
        return
    
    logger.info(f"找到 {len(inpaths)} 个文件")
    
    for file_idx, current_path in enumerate(inpaths):
        logger.info(f"[{file_idx + 1}/{len(inpaths)}] 处理文件: {current_path}")
        
        if not current_path.exists():
            logger.error(f"文件不存在: {current_path}")
            continue
            
        try:
            # 使用 with 确保文件正确关闭
            with xopen.xopen(current_path, "rt", encoding=encoding) as fin:
                for line_no, line in enumerate(fin, 1):
                    line = line.strip()
                    
                    # 跳过空行
                    if not line:
                        continue
                    
                    try:
                        data = orjson.loads(line)
                        yield current_path, data
                        
                    except orjson.JSONDecodeError as e:
                        if skip_errors:
                            logger.warning(
                                f"跳过无效 JSON (文件: {current_path.name}, "
                                f"行: {line_no}): {str(e)[:100]}"
                            )
                            continue
                        else:
                            raise ValueError(
                                f"JSON 解析失败 (文件: {current_path.name}, 行: {line_no}): {e}"
                            ) from e
                    
                    except Exception as e:
                        logger.error(f"处理行 {line_no} 时出错: {e}")
                        if not skip_errors:
                            raise
                            
        except Exception as e:
            logger.error(f"读取文件失败 {current_path}: {e}")
            if not skip_errors:
                raise

                
from cetc_product.data_model.entity import EntityType, DomainEnum, RegionEnum
target_domains = {d.value for d in DomainEnum.get_target_domains()}
target_regions = {r.value for r in RegionEnum.get_target_regions()}
def is_task1(record):
    entity_info = record.get("entity_classification", {}).get("result",{})
    if entity_info.get("type") != EntityType.ORGANIZATION.value:
        return False
    domains = safe_get(record, ["domain_and_region_classifier", "result", "domains"], None)
    if domains is None or (not(set(domains) & target_domains)):
        return False
    
    regions = safe_get(record, ["domain_and_region_classifier", "result", "regions"], None)
    if regions is None or (not(set(regions) & target_regions))  :
        return False
    return True

def get_input(data):
    infobox_info = safe_get(data, ["infoboxes_info"], None)
    if not infobox_info:
        infobox_info = ""
    else:
        infobox_info = yaml.safe_dump(infobox_info[0], allow_unicode=True, sort_keys=False)
    
    raw_text_path = Path(data.get("markdown_path",None))
    if raw_text_path.exists():
        fragment = raw_text_path.read_text(encoding="utf-8")[:1000]
    else:
        fragment = f"# {data['title']}\n\n{data['abstract']}"
    return {"wiki_text" : f"<infobox>{infobox_info}</infobox>\n\n{fragment}"}

In [ ]:
# 准备提示词
from cetc_product.data_model.NER_for_org import OrganizationInfo
from cetc_product.tools.load_tool import load_prompt
prompt_path = "/Data_two/wyw/code/CETC_product/data/prompts/NER_for_org.txt"

prompt_temp = load_prompt(prompt_path, schema_define_cls=OrganizationInfo)

In [ ]:
prompt_temp.pretty_print()

In [ ]:
def extract_info(record, task):
    if task == "task2":
        raise NotImplementedError
    

In [ ]:
Path("/mnt/1.json").suffix

In [ ]:
# 加载模型
from cetc_product.tools.llm_factory import llm_manager

In [ ]:
from cetc_product.tools.json_parser import MyJSONParser


main_chain = prompt_temp | llm_manager["gpt_low"] | MyJSONParser(OrganizationInfo)

In [ ]:
inpath = Path(config["DATA"]["IN"]["zhwiki_inpath"])
for p, data in read_indata(inpath):
    if is_task1(data):
        extract_result = main_chain.invoke(get_input(data))#extract_info(data, "task1")
        break
    
    # elif is_task2(data):
    #     extract_result = extract_info(data, "task2")
    
    ## OK 先对其进行提取
    ...

In [ ]:
extract_result

In [ ]:
data.keys()